# 三模型投票的固定测试集评估

沿用 Notebook 022 的模型和规则。按 B06、TextCNN、DistilBERT 的顺序，加权软投票固定为 (0.2, 0.3, 0.5)，等权软投票取平均值，两种软投票都以正类概率严格大于 0.5 判为正类。硬投票使用三个模型原有的类别预测，至少两票判为正类。

复用 Notebook 015 和 020 已保存的测试预测，对同一批样本补算校准后 B06 的概率。三种方案共用清洗后的测试集，全程固定模型、阈值和权重。

In [ ]:
# 按固定协议计算三种投票结果
import hashlib
import json

from sentiment_analysis.paths import TABLES_DIR
from sentiment_analysis.voting import VOTING_PROTOCOL_PATH, evaluate_fixed_voting, save_voting_results

protocol_bytes = VOTING_PROTOCOL_PATH.read_bytes()
print(f"Protocol SHA-256: {hashlib.sha256(protocol_bytes).hexdigest()}")
comparison, prediction_table = evaluate_fixed_voting(json.loads(protocol_bytes))
print(f"Aligned test samples: {len(prediction_table):,}")
display(comparison.round(4))

In [ ]:
# 保存指标和逐样本投票结果
save_voting_results(TABLES_DIR, comparison, prediction_table)

## 测试结果

| 方案 | 准确率 | F1 | 相对硬投票的 F1 变化 | 纠正错误 | 新增错误 |
| --- | ---: | ---: | ---: | ---: | ---: |
| 硬投票 | 91.37% | 91.45% | +0.00 个百分点 | 0 | 0 |
| 等权软投票 | 92.01% | 92.07% | +0.62 个百分点 | 333 | 176 |
| 加权软投票 | 92.15% | 92.25% | +0.79 个百分点 | 504 | 311 |

在 24,678 条测试样本上，两种软投票都超过了硬投票。固定加权比等权再高 0.18 个 F1 百分点，相比硬投票净减少 193 个错误。测试集此前用于其他固定模型的评估，这次的投票规则依据验证集结果事先固定。
